# FastLp Phase 4: Complete, resumable A100 experiments

**Goal:** Run all 6 combinations (3 exponents x 2 matrix sizes), each with 3 seeds and the same three proximal backends. After every successful case, archive the original Phase-3 results and the additional Phase-4 diagnostics. Finally, combine the results, audit them, and download one ZIP.

**Read first**

- In Colab, choose **Runtime > Change runtime type > GPU (A100)** if available.
- **Run all cells from top to bottom.** This notebook does not depend on the current working directory (`/content` is fine).
- **Google Drive backup is ON by default.** Authorize the Drive mount when prompted. Each completed case is saved as a separate ZIP in Drive, so a later Colab session can resume rather than rerun completed cases. This is important because `/content` is ephemeral.
- The notebook pins the code to **`2dec97b158f8f4d935616ee1842ca5f5f8ebff5d`**, the commit previously used in the successful pilot. It refuses to run formal A100 timings on a different revision.
- Original Phase-3 outputs are not modified by the Phase-4 diagnostics. The archived measurements and the new instrumented per-stage timings have different timing semantics.
- If GPU is not an A100, the notebook intentionally stops unless you explicitly change `REQUIRE_A100`.

### Six planned cases

| Exponent | Matrix (m, n) | Methods |
|---|---|---|
| 1/3 | (1000, 2000), (2000, 4000) | Chebyshev D6, Newton K3, Bisection K20 |
| 3/4 | (1000, 2000), (2000, 4000) | Chebyshev D12, Newton K4, Bisection K20 |
| 0.9 | (1000, 2000), (2000, 4000) | Chebyshev D24, Newton K4, Bisection K20 |

Settings: three seeds (0, 1, 2), 500 maximum iterations, `verify_every=10`, independently verified tolerance `1e-6`, float32 online backends, and separate high-accuracy reference checks.

## 1. Configuration

For reliable recovery after disconnection, keep `SAVE_TO_DRIVE=True`. The Drive folder is unique to the pinned commit and may be reused when reconnecting; completed cases are restored from ZIP checkpoints. To do a clean, separate campaign later, change `DRIVE_RUN_NAME` and `LOCAL_RESULT_NAME` **together**.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Jy-U1/Fast_Lp_Prox.git"
BRANCH = "revision/major-experiments"
PINNED_SHA = "2dec97b158f8f4d935616ee1842ca5f5f8ebff5d"

REPO_DIR = Path("/content/Fast_Lp_Prox")
LOCAL_RESULT_NAME = "reviewer2_phase4_lasso_all"
RESULT_ROOT = REPO_DIR / "results" / LOCAL_RESULT_NAME

SAVE_TO_DRIVE = True
DRIVE_RUN_NAME = "FastLp_Phase4_A100_All6_2dec97b1"
DRIVE_ROOT = Path("/content/drive/MyDrive") / DRIVE_RUN_NAME

REQUIRE_A100 = True
RUN_SMOKE_TESTS = True
STAGE_REPEATS = 3
STAGE_WARMUP = 1
DOWNLOAD_FINAL_ZIP = True

CASES = [
    {"tag": "p03333_m1000_n2000", "p": "1/3", "m": 1000, "n": 2000,
     "methods": ("chebyshev_D6", "newton_K3", "bisection_K20")},
    {"tag": "p03333_m2000_n4000", "p": "1/3", "m": 2000, "n": 4000,
     "methods": ("chebyshev_D6", "newton_K3", "bisection_K20")},
    {"tag": "p07500_m1000_n2000", "p": "3/4", "m": 1000, "n": 2000,
     "methods": ("chebyshev_D12", "newton_K4", "bisection_K20")},
    {"tag": "p07500_m2000_n4000", "p": "3/4", "m": 2000, "n": 4000,
     "methods": ("chebyshev_D12", "newton_K4", "bisection_K20")},
    {"tag": "p09000_m1000_n2000", "p": "0.9", "m": 1000, "n": 2000,
     "methods": ("chebyshev_D24", "newton_K4", "bisection_K20")},
    {"tag": "p09000_m2000_n4000", "p": "0.9", "m": 2000, "n": 4000,
     "methods": ("chebyshev_D24", "newton_K4", "bisection_K20")},
]

print("Repository:", REPO_DIR)
print("Output:", RESULT_ROOT)
print("Drive backup enabled:", SAVE_TO_DRIVE)
print("Number of cases:", len(CASES))


## 2. Mount Drive, clone the exact code revision, and verify A100

Only the pinned commit is used. This cell never deletes an existing repository or silently resets tracked changes. Re-running it in the same session is safe. The Google Drive copy is a separate checkpoint store; Phase-3 code itself still runs against the fast local `/content` filesystem.

In [ ]:
import os
import sys
import json
import shutil
import subprocess
import platform
from datetime import datetime, timezone


def run_cmd(cmd, cwd=None, capture=False):
    print("$", " ".join(str(x) for x in cmd), flush=True)
    if capture:
        return subprocess.check_output([str(x) for x in cmd], cwd=cwd, text=True).strip()
    subprocess.run([str(x) for x in cmd], cwd=cwd, check=True)


if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("Google Drive mount requires Google Colab. Set SAVE_TO_DRIVE=False for local-only use.") from exc
    drive.mount("/content/drive")
    (DRIVE_ROOT / "case_checkpoints").mkdir(parents=True, exist_ok=True)
    (DRIVE_ROOT / "final_archives").mkdir(parents=True, exist_ok=True)
    print("Persistent Drive backup:", DRIVE_ROOT)
else:
    print("WARNING: Drive backup OFF; /content results can disappear after runtime reset.")

if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(f"Directory exists but is not a git clone: {REPO_DIR}")
    run_cmd(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR])

current_sha = run_cmd(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture=True)
if current_sha != PINNED_SHA:
    dirty = run_cmd(["git", "status", "--porcelain"], cwd=REPO_DIR, capture=True)
    if dirty:
        raise RuntimeError("Repository has uncommitted changes. Cannot switch code revision safely.")
    run_cmd(["git", "fetch", "origin", PINNED_SHA], cwd=REPO_DIR)
    run_cmd(["git", "checkout", "--detach", PINNED_SHA], cwd=REPO_DIR)

actual_sha = run_cmd(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture=True)
assert actual_sha == PINNED_SHA, (actual_sha, PINNED_SHA)
assert (REPO_DIR / "revision/phase4/scripts/run_reviewer2_lasso_matrix.py").is_file()
assert (REPO_DIR / "revision/phase4/scripts/diagnose_lasso_outputs.py").is_file()
print("Pinned Git SHA:", actual_sha)

run_cmd([sys.executable, "-m", "pip", "install", "-e", ".[dev]"], cwd=REPO_DIR)

import torch
import pandas as pd
assert torch.cuda.is_available(), "CUDA is unavailable. Select a GPU runtime."
gpu_name = torch.cuda.get_device_name(0)
print("GPU:", gpu_name)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
run_cmd(["nvidia-smi"])
if REQUIRE_A100 and "A100" not in gpu_name.upper():
    raise RuntimeError(f"Expected A100 for formal timing comparisons; got {gpu_name}")

if RUN_SMOKE_TESTS:
    run_cmd([
        sys.executable, "-m", "pytest", "-q",
        "tests/test_phase3_lasso.py",
        "tests/test_phase4_reporting.py",
        "tests/test_phase4_lasso_diagnostics.py",
    ], cwd=REPO_DIR)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print("Environment ready.")


## 3. Recovery and verification helpers

These helpers inspect expected files, method/seed coverage, numerical replay checks, and metadata. A case with a stopping or replay warning can still be **preserved** for investigation, but it is clearly marked as not fully verified. No existing experimental directory is overwritten.

In [ ]:
import hashlib
import math
import zipfile

CHECKPOINT_TMP = Path("/content/fastlp_phase4_checkpoints")
CHECKPOINT_TMP.mkdir(parents=True, exist_ok=True)


def bool_count(series):
    return int(series.astype(str).str.strip().str.lower().isin(["true", "1", "1.0"]).sum())


def case_paths(case):
    base = RESULT_ROOT / case["tag"]
    return base, base / "phase3_results", base / "phase4_diagnostics"


def inspect_case(case):
    base, p3, p4 = case_paths(case)
    required = [
        p3 / "metadata.json", p3 / "run_summary.csv", p3 / "trajectories.csv",
        p3 / "reference_FB_trajectories.csv", p3 / "timings_raw.csv",
        p4 / "diagnostics_summary.csv", p4 / "recovery_trajectories.csv",
        p4 / "stage_timings_raw.csv", p4 / "diagnostics_metadata.json",
    ]
    missing = [str(p.relative_to(base)) for p in required if not p.is_file()]
    if missing:
        return {"tag": case["tag"], "complete": False, "missing": missing}
    try:
        s = pd.read_csv(p3 / "run_summary.csv")
        d = pd.read_csv(p4 / "diagnostics_summary.csv")
        r = pd.read_csv(p4 / "recovery_trajectories.csv")
        meta = json.loads((p3 / "metadata.json").read_text())
        dmeta = json.loads((p4 / "diagnostics_metadata.json").read_text())
        pairs = {(int(a), str(b)) for a, b in zip(s["seed"], s["method"])}
        dpairs = {(int(a), str(b)) for a, b in zip(d["seed"], d["method"])}
        expected = {(seed, method) for seed in [0, 1, 2] for method in case["methods"]}
        structural = (
            len(s) == 9 and len(d) == 9
            and pairs == expected and dpairs == expected
            and meta.get("code_commit") == PINNED_SHA
            and dmeta.get("original_phase3_commit") == PINNED_SHA
            and int(meta.get("m", -1)) == case["m"]
            and int(meta.get("n", -1)) == case["n"]
            and len(r) > 0
        )
        stops = bool_count(s["verified_stop_hit"])
        replay = bool_count(d["phase3_replay_check_pass"])
        return {
            "tag": case["tag"], "complete": bool(structural),
            "stop_hits": stops, "replay_passes": replay,
            "all_stop": stops == 9, "all_replay": replay == 9,
            "rows_phase3": len(s), "rows_diagnostics": len(d),
            "git_sha": meta.get("code_commit"),
        }
    except Exception as exc:
        return {"tag": case["tag"], "complete": False, "error": str(exc)}


def phase3_is_ready(case):
    _, p3, _ = case_paths(case)
    core = ["metadata.json", "run_summary.csv", "trajectories.csv"]
    if any(not (p3 / item).exists() for item in core):
        return False
    try:
        s = pd.read_csv(p3 / "run_summary.csv")
        meta = json.loads((p3 / "metadata.json").read_text())
        return len(s) == 9 and meta.get("code_commit") == PINNED_SHA
    except Exception:
        return False


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def checkpoint_path(case):
    return DRIVE_ROOT / "case_checkpoints" / (case["tag"] + ".zip")


def save_case_checkpoint(case):
    base, _, _ = case_paths(case)
    report = inspect_case(case)
    if not report.get("complete"):
        raise RuntimeError(f"Cannot checkpoint an incomplete case: {report}")

    local_zip = CHECKPOINT_TMP / (case["tag"] + ".zip")
    with zipfile.ZipFile(local_zip, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for file in sorted(base.rglob("*")):
            if file.is_file():
                zf.write(file, file.relative_to(RESULT_ROOT))
    with zipfile.ZipFile(local_zip) as zf:
        assert zf.testzip() is None

    if SAVE_TO_DRIVE:
        target = checkpoint_path(case)
        if target.exists():
            with zipfile.ZipFile(target) as zf:
                if zf.testzip() is not None:
                    raise RuntimeError(f"Existing Drive checkpoint has corrupt contents: {target}")
            print("Existing Drive checkpoint preserved:", target.name)
        else:
            staging = target.with_suffix(".zip.part")
            shutil.copy2(local_zip, staging)
            with zipfile.ZipFile(staging) as zf:
                assert zf.testzip() is None
            os.replace(staging, target)
            print("Checkpoint saved to Drive:", target.name)
    else:
        print("Local checkpoint:", local_zip)


def restore_drive_checkpoint(case):
    if not SAVE_TO_DRIVE:
        return False
    saved = checkpoint_path(case)
    if not saved.is_file():
        return False
    base, _, _ = case_paths(case)
    if base.exists() and inspect_case(case).get("complete"):
        return True
    if base.exists():
        quarantine_incomplete_directory(base)
    with zipfile.ZipFile(saved) as zf:
        bad = zf.testzip()
        if bad is not None:
            raise RuntimeError(f"Corrupt Drive checkpoint: {saved}, first bad file: {bad}")
        for member in zf.namelist():
            parts = Path(member).parts
            if not parts or parts[0] != case["tag"] or ".." in parts:
                raise RuntimeError(f"Unexpected file in checkpoint {saved}: {member}")
        zf.extractall(RESULT_ROOT)
    report = inspect_case(case)
    if not report.get("complete"):
        raise RuntimeError(f"Drive checkpoint does not pass validation: {report}")
    print("Restored from Google Drive:", case["tag"])
    return True


def quarantine_incomplete_directory(path):
    if not path.exists():
        return None
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    target = path.with_name(path.name + "__interrupted_" + timestamp)
    idx = 0
    while target.exists():
        idx += 1
        target = path.with_name(path.name + "__interrupted_" + timestamp + f"_{idx}")
    path.rename(target)
    print("Preserved interrupted output at:", target)
    return target


def record_progress(reports, errors):
    info = {
        "updated_utc": datetime.now(timezone.utc).isoformat(),
        "git_commit": PINNED_SHA,
        "expected_cases": [c["tag"] for c in CASES],
        "cases": reports,
        "errors": errors,
    }
    progress_path = RESULT_ROOT / "campaign_progress.json"
    progress_path.write_text(json.dumps(info, indent=2, default=str) + "\n")
    if SAVE_TO_DRIVE:
        dest = DRIVE_ROOT / "campaign_progress.json"
        temp = DRIVE_ROOT / "campaign_progress.json.part"
        shutil.copy2(progress_path, temp)
        os.replace(temp, dest)
    return info

print("Checkpoint and validation helpers ready.")


## 4. Execute all six cases with per-case checkpointing

This is the main experimental cell. Each case calls the **existing** Phase-3 driver with the same CLI as before, then calls the **existing** Phase-4 diagnostics. The loop copies a separate ZIP to Drive **immediately after each verified structural completion**. If Colab disconnects later, rerun this notebook from the top; completed cases will be restored and skipped.

A case that fails the independent stop criterion, or fails a replay check, is still retained for scientific audit. Such a case will be flagged rather than presented as a successful convergence result. Interrupted partial outputs are renamed and kept instead of deleted.

In [ ]:
from IPython.display import display

case_reports = []
case_errors = []
STARTED_UTC = datetime.now(timezone.utc).isoformat()

for index, case in enumerate(CASES, start=1):
    print("\n" + "=" * 76, flush=True)
    print(f"CASE {index}/{len(CASES)}: {case['tag']} (p={case['p']}, m={case['m']}, n={case['n']})", flush=True)
    print("=" * 76, flush=True)
    base, p3, p4 = case_paths(case)

    try:
        previous = inspect_case(case)
        if not previous.get("complete"):
            restored = restore_drive_checkpoint(case)
            if restored:
                previous = inspect_case(case)
        if previous.get("complete"):
            print("Reusing fully generated case:", case["tag"])
        else:
            if not phase3_is_ready(case):
                if base.exists():
                    quarantine_incomplete_directory(base)
                run_cmd([
                    sys.executable, "revision/phase4/scripts/run_reviewer2_lasso_matrix.py",
                    "--mode", "formal", "--device", "cuda",
                    "--only-p", case["p"],
                    "--only-size", f"{case['m']}x{case['n']}",
                    "--output-root", str(RESULT_ROOT),
                    "--execute",
                ], cwd=REPO_DIR)
            else:
                print("Valid Phase-3 output already exists; only diagnostics needed.")

            if p4.exists() and not inspect_case(case).get("complete"):
                quarantine_incomplete_directory(p4)
            if not p4.exists():
                run_cmd([
                    sys.executable, "-m", "revision.phase4.scripts.diagnose_lasso_outputs",
                    "--input-dir", str(p3),
                    "--device", "cuda",
                    "--stage-repeats", str(STAGE_REPEATS),
                    "--stage-warmup", str(STAGE_WARMUP),
                ], cwd=REPO_DIR)

        result = inspect_case(case)
        if not result.get("complete"):
            raise RuntimeError(f"Incomplete case after running diagnostics: {result}")
        print("STOP:", f"{result['stop_hits']}/9", "REPLAY:", f"{result['replay_passes']}/9")
        if not result["all_stop"] or not result["all_replay"]:
            print("WARNING: The case completed but has numerical warnings; inspect it before claiming success.")
        save_case_checkpoint(case)
        case_reports.append(result)
    except Exception as exc:
        case_errors.append({"tag": case["tag"], "error": repr(exc)})
        print("ERROR in", case["tag"], ":", repr(exc), flush=True)
        print("Earlier completed cases remain backed up. Continuing to other cases.")
    finally:
        record_progress(case_reports, case_errors)

print("\n--- CAMPAIGN EXECUTION FINISHED ---")
print("Complete structural cases:", sum(x.get("complete", False) for x in case_reports), "/", len(CASES))
print("Numerical failures:", sum(not (x.get("all_stop") and x.get("all_replay")) for x in case_reports))
print("Execution errors:", len(case_errors))
if case_errors:
    for item in case_errors:
        print(item)


## 5. Consolidate results at the **correct** iteration checkpoints

The original diagnostic summary records recovery quality at the last replayed iteration (often 500), whereas Phase-3 runtime is measured up to the first verified stop (e.g., 130/150/160). To avoid confusing them, this cell joins the original `recovery_trajectories.csv` to **`first_verified_stop_iteration`** and saves both at-stop and final-checkpoint metrics separately.

It also keeps the instrumented stage timings separate from the uninstrumented replay/checked-stop timings, and calculates a descriptive offline-table amortization estimate only when the Chebyshev method is faster than Newton under the tested settings.

In [ ]:
import pandas as pd

ANALYSIS_DIR = REPO_DIR / "results" / "reviewer2_phase4_all6_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

all_summaries = []
all_diagnostics = []
all_recovery_at_stop = []
offline_rows = []
inspection = []

for case in CASES:
    report = inspect_case(case)
    inspection.append(report)
    if not report.get("complete"):
        print("MISSING/INCOMPLETE:", case["tag"], report)
        continue
    base, p3, p4 = case_paths(case)
    s = pd.read_csv(p3 / "run_summary.csv")
    d = pd.read_csv(p4 / "diagnostics_summary.csv")
    r = pd.read_csv(p4 / "recovery_trajectories.csv")
    meta = json.loads((p3 / "metadata.json").read_text())

    for frame in (s, d):
        frame.insert(0, "case", case["tag"])
        frame.insert(1, "m_size", case["m"])
        frame.insert(2, "n_size", case["n"])
        frame.insert(3, "p_label", case["p"])

    s_for_join = s[["seed", "method", "first_verified_stop_iteration"]].copy()
    stop = s_for_join.merge(
        r,
        left_on=["seed", "method", "first_verified_stop_iteration"],
        right_on=["seed", "method", "iteration"],
        how="left", validate="one_to_one",
    )
    stop.insert(0, "case", case["tag"])
    stop.insert(1, "m_size", case["m"])
    stop.insert(2, "n_size", case["n"])
    stop.insert(3, "p_label", case["p"])
    if stop["recovery_l2_relative"].isna().any():
        print("WARNING: recovery not available at a stopping iteration in", case["tag"])

    all_summaries.append(s)
    all_diagnostics.append(d)
    all_recovery_at_stop.append(stop)
    offline_rows.append({
        "case": case["tag"], "p": case["p"],
        "m": case["m"], "n": case["n"],
        "offline_fit_seconds": meta.get("offline_fit_seconds"),
        "code_commit": meta.get("code_commit"),
        "gpu": meta.get("gpu"),
    })

if not all_summaries:
    raise RuntimeError("No complete cases available for aggregation. Inspect campaign errors above.")

S = pd.concat(all_summaries, ignore_index=True)
D = pd.concat(all_diagnostics, ignore_index=True)
RSTOP = pd.concat(all_recovery_at_stop, ignore_index=True)
OFFLINE = pd.DataFrame(offline_rows)

S.to_csv(ANALYSIS_DIR / "all_phase3_run_summaries.csv", index=False)
D.to_csv(ANALYSIS_DIR / "all_phase4_diagnostics_final.csv", index=False)
RSTOP.to_csv(ANALYSIS_DIR / "all_recovery_at_first_verified_stop.csv", index=False)
OFFLINE.to_csv(ANALYSIS_DIR / "offline_fit_costs.csv", index=False)

summary_rows = []
for case in CASES:
    s = S[S["case"] == case["tag"]]
    d = D[D["case"] == case["tag"]]
    r = RSTOP[RSTOP["case"] == case["tag"]]
    if s.empty:
        continue
    cheb = s[s["method"].str.startswith("chebyshev_")]
    newt = s[s["method"].str.startswith("newton_")]
    offline_s = OFFLINE.loc[OFFLINE["case"] == case["tag"], "offline_fit_seconds"].iloc[0]
    saving_ms = float(newt["replay_budget_median_ms"].mean() - cheb["replay_budget_median_ms"].mean())
    breakeven = math.ceil(float(offline_s) * 1000 / saving_ms) if saving_ms > 0 and pd.notna(offline_s) else None

    for method in case["methods"]:
        sm = s[s["method"] == method]
        dm = d[d["method"] == method]
        rm = r[r["method"] == method]
        summary_rows.append({
            "case": case["tag"], "p": case["p"], "m": case["m"], "n": case["n"],
            "method": method, "n_seeds": len(sm),
            "n_stopping_success": bool_count(sm["verified_stop_hit"]),
            "n_replay_success": bool_count(dm["phase3_replay_check_pass"]),
            "mean_first_stop_iteration": sm["first_verified_stop_iteration"].mean(),
            "mean_uninstrumented_replay_ms": sm["replay_budget_median_ms"].mean(),
            "mean_checked_stop_wall_ms": sm["checked_stop_median_wall_ms"].mean(),
            "mean_relative_l2_recovery_at_stop": rm["recovery_l2_relative"].mean(),
            "mean_relative_l2_recovery_at_final_checkpoint": dm["recovery_l2_relative"].mean(),
            "mean_support_precision_at_stop": rm["support_precision"].mean(),
            "mean_support_recall_at_stop": rm["support_recall"].mean(),
            "mean_active_coordinate_fraction_at_stop": rm["active_coordinate_fraction"].mean(),
            "mean_instrumented_gradient_forward_ms": dm["instrumented_gradient_forward_median_ms"].mean(),
            "mean_instrumented_prox_ms": dm["instrumented_prox_median_ms"].mean(),
            "offline_fit_ms_chebyshev_only": float(offline_s) * 1000 if method.startswith("chebyshev_") else float("nan"),
            "illustrative_break_even_solves_chebyshev_vs_newton": breakeven if method.startswith("chebyshev_") else None,
        })

SUMMARY = pd.DataFrame(summary_rows)
SUMMARY.to_csv(ANALYSIS_DIR / "all6_method_summary.csv", index=False)

print("Methods summarized:", len(SUMMARY), "(expected 18 when all 6 cases are present)")
print("Per-seed records:", len(S), "(expected 54 when all 6 cases are present)")
display(SUMMARY[[
    "case", "method", "n_stopping_success", "n_replay_success",
    "mean_first_stop_iteration", "mean_uninstrumented_replay_ms",
    "mean_relative_l2_recovery_at_stop",
]].round(5))


## 6. Final self-audit, ZIP archive, persistent backup, and download

This step packages every available case directory (including Phase-3 raw trajectories, original timings, Phase-4 diagnostics, launch logs, and source metadata) plus the aggregated CSVs. It embeds a JSON manifest and SHA-256 file hashes. If any case is missing or any numerical check fails, the file is labeled `INCOMPLETE` and the manifest explains the issue; **no fictitious success** is recorded.

The complete ZIP is also written to Google Drive before the browser download starts. A failed browser download therefore does **not** imply that the data is lost.

In [ ]:
import zipfile

FINISHED_UTC = datetime.now(timezone.utc).isoformat()
case_status = [inspect_case(case) for case in CASES]
per_case_environment = {}
for case in CASES:
    base, phase3_dir, _ = case_paths(case)
    meta_path = phase3_dir / "metadata.json"
    if meta_path.is_file():
        meta = json.loads(meta_path.read_text())
        per_case_environment[case["tag"]] = {
            "gpu": meta.get("gpu"),
            "python": meta.get("python"),
            "torch": meta.get("torch"),
            "numpy": meta.get("numpy"),
            "device": meta.get("device"),
        }
unique_environments = {json.dumps(v, sort_keys=True) for v in per_case_environment.values()}
same_environment = len(unique_environments) <= 1
full_campaign = all(
    s.get("complete") and s.get("all_stop") and s.get("all_replay")
    for s in case_status
) and len(case_errors) == 0 and same_environment

status_name = "COMPLETE" if full_campaign else "INCOMPLETE"
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
final_zip = Path("/content") / f"FastLp_Phase4_A100_All6_{PINNED_SHA[:8]}_{status_name}_{stamp}.zip"

file_entries = []
files_to_archive = []
for case in CASES:
    base, _, _ = case_paths(case)
    if base.is_dir():
        for file in sorted(base.rglob("*")):
            if file.is_file():
                name = Path(LOCAL_RESULT_NAME) / file.relative_to(RESULT_ROOT)
                files_to_archive.append((file, name))
for file in sorted(ANALYSIS_DIR.rglob("*")):
    if file.is_file():
        name = Path("analysis") / file.relative_to(ANALYSIS_DIR)
        files_to_archive.append((file, name))
progress_file = RESULT_ROOT / "campaign_progress.json"
if progress_file.exists():
    files_to_archive.append((progress_file, Path("campaign_progress.json")))

for file, name in files_to_archive:
    file_entries.append({
        "archive_path": str(name), "bytes": file.stat().st_size,
        "sha256": sha256_file(file),
    })

manifest = {
    "project": "FastLp Phase 4 reviewer Lasso six-case campaign",
    "status": status_name,
    "started_utc": STARTED_UTC,
    "finished_utc": FINISHED_UTC,
    "git_commit": PINNED_SHA,
    "git_status": run_cmd(["git", "status", "--porcelain"], cwd=REPO_DIR, capture=True),
    "hardware": gpu_name,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "cases": case_status,
    "per_case_environment": per_case_environment,
    "same_environment_across_cases": same_environment,
    "errors": case_errors,
    "notes": [
        "First-stop recovery comes from recovery_trajectories.csv at first_verified_stop_iteration.",
        "Final-checkpoint recovery and time-to-tolerance may refer to different iteration counts.",
        "Instrumented gradient/prox times are diagnostic and need not sum to uninstrumented replay time.",
        "A fixed nonzero polynomial error is not a proof of exact critical-point convergence.",
    ],
    "file_checksums": file_entries,
}
manifest_json = json.dumps(manifest, indent=2, default=str)
(ANALYSIS_DIR / "final_campaign_manifest.json").write_text(manifest_json + "\n")

with zipfile.ZipFile(final_zip, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for file, archive_name in files_to_archive:
        zf.write(file, str(archive_name))
    zf.writestr("archive_manifest.json", manifest_json)

with zipfile.ZipFile(final_zip) as zf:
    assert zf.testzip() is None, "ZIP integrity test failed"
    print("Archived entries:", len(zf.namelist()))

print("\nFinal status:", status_name)
if not same_environment:
    print("WARNING: Software or GPU environments differ across cases; review metadata before using timings.")
print("Case reports:")
for item in case_status:
    print(" ", item["tag"], "complete=", item.get("complete"),
          "stop=", item.get("stop_hits"), "replay=", item.get("replay_passes"))
print("Final archive:", final_zip)
print("Size (MB):", round(final_zip.stat().st_size / (1024 ** 2), 2))

if SAVE_TO_DRIVE:
    target = DRIVE_ROOT / "final_archives" / final_zip.name
    staging = target.with_suffix(".zip.part")
    shutil.copy2(final_zip, staging)
    with zipfile.ZipFile(staging) as zf:
        assert zf.testzip() is None
    os.replace(staging, target)
    print("Persistent Google Drive copy:", target)

if DOWNLOAD_FINAL_ZIP:
    try:
        from google.colab import files
        files.download(str(final_zip))
    except Exception as exc:
        print("Automatic browser download did not start:", repr(exc))
        print("The ZIP remains available at:", final_zip)
        if SAVE_TO_DRIVE:
            print("You can also download it from Google Drive:", target)


## Notes on interruption and reporting

- **After a Colab disconnect:** rerun this notebook from the top with the same `DRIVE_RUN_NAME` and `LOCAL_RESULT_NAME`. It restores verified per-case ZIP checkpoints from Google Drive. Without Drive backup, Colab cannot recover files that were removed when the runtime reset.
- **If a case fails:** earlier checkpoint ZIPs remain in Drive. Rerunning the notebook retries incomplete cases without overwriting original completed ones. Interrupted local directories are preserved under `__interrupted_*` names.
- **For the manuscript:** use the `analysis/all6_method_summary.csv` and per-case raw CSVs together. A positive `verified_stop_hit` is a *numerical* verified-displacement check, not a mathematical proof of convergence to an exact critical point. Do not label fixed-budget timings as time-to-tolerance for cases without a stop hit.
- **Important scope:** this notebook addresses Reviewer 2's multi-exponent/multi-size Lasso request. It does not automatically implement the certified-hybrid scalar baseline or the optional objective safeguard; those are separate research/code tasks.